# QC confound 05 — Is library depth the PBS signal in every cancer type?

Notebook 04 asked this for ER+ versus Normal. This notebook repeats that
check once for every other `cancer_type` against Normal: BRCA1 pre-neoplastic,
HER2+, triple negative, triple negative BRCA1, and PR+.

The procedure matches notebook 04. Discarded cells are redefined with the
per-specimen MAD rule. Libraries are thinned to the shallowest included
specimen in that contrast. The PBS classifier is the same one-hot of
discarded cells, and the specimen is the unit. A type with fewer than two
specimens after the published cell minimum is skipped.

The target depth is chosen inside each contrast, so it is not one number
shared by every cancer type. The ER+ row is the notebook 04 comparison.

This loads the full Total-population study once, then thins and refits each
contrast. It is the slow notebook in the series.

In [ ]:
import logging

import pandas as pd
from IPython.display import Markdown, display

from signals_in_the_noise.analysis.depth_by_subtype import (
    depth_by_subtype_handoff,
    run_depth_by_subtype,
)
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.preprocessing.gse161529 import GSE161529
from signals_in_the_noise.preprocessing.specimens import Cohort
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)

OUTPUT_DIR = get_data_path("processed") / "qc_confound"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

In [ ]:
gse = GSE161529()
totals = [
    adata for adata in gse.objects.values() if str(adata.uns.get("cell_population")) == "Total"
]
cohort = Cohort.from_objects(totals, sorted({str(adata.uns["cancer_type"]) for adata in totals}))
cohort.overview().groupby("condition").agg(specimens=("n_cells", "size"), cells=("n_cells", "sum"))

## Depth check for each cancer type

`reading` labels the specimen permutation p-value at 0.05. Collapse after
thinning is the depth explanation found for ER+. The p-values themselves are
in the table.

In [ ]:
result = run_depth_by_subtype(
    cohort.specimens, cohort.conditions, cohort.patients, n_permutations=200, seed=0
)
result.to_csv(OUTPUT_DIR / "05_depth_by_subtype.csv", index=False)
handoff = depth_by_subtype_handoff(result)
(OUTPUT_DIR / "05_handoff.md").write_text(handoff, encoding="utf-8")
display(result)
display(Markdown(handoff))